# Single Cell data
CC 2026-08-11

## 1. Setup

In [ ]:
import retinanalysis as ra
import matplotlib.pyplot as plt
import numpy as np

# Read-only single-cell database queries and notebook browsers.
from retinanalysis.SCutils import explore as sc
# Map new h5 files when needed.
report = ra.SCutils.update_single_cell_json()

Single-cell drive: /Volumes/ChrisNewSSD
H5 folder: /Volumes/ChrisNewSSD/single_cell/chris_data/h5
JSON folder: /Volumes/ChrisNewSSD/single_cell/chris_data/json
New H5 files (3): 2026-08-10_G.h5, 2026-08-12_G.h5, 2026-08-12_G_2.h5
Parsing 2026-08-10_G.h5 -> 2026-08-10_G.json
Parsing experiment sources.
Parsing group 1 of 18...
Parsing group: Control
Parsing group 2 of 18...
Parsing group: Control
Parsing group 3 of 18...
Parsing group: Control
Parsing group 4 of 18...
Parsing group: Control
Parsing group 5 of 18...
Parsing group: Control
Parsing group 6 of 18...
Parsing group: Control
Parsing group 7 of 18...
Parsing group: Control
Parsing group 8 of 18...
Parsing group: Control
Parsing group 9 of 18...
Parsing group: Control
Parsing group 10 of 18...
Parsing group: Control
Parsing group 11 of 18...
Parsing group: Control
Parsing group 12 of 18...
Parsing group: Control
Parsing group 13 of 18...
Parsing group: Control
Parsing group 14 of 18...
Parsing group: Control
Parsing group 15 of 

## 2. Populate and refresh the database

`populate_database()` ingests new experiments, refreshes experiments whose JSON changed, and returns the database freshness check in the same report. Canonical experiment names such as `YYYY-MM-DD_X.h5` are included; auxiliary and legacy files are ignored. By default only metadata and tags JSON files trigger a refresh. Pass `watch_data_file=True` to include H5 modification times.

In [3]:
# One call handles ingest, refresh, and the post-ingest stale-file check.
summary = ra.populate_database()
df_db = summary['experiments']
df_stale = summary['stale']

print(f"newly added : {len(summary['added'])}")
print(f"refreshed   : {len(summary['updated'])}")
print(f"errored     : {len(summary['skipped'])}")
print(f"database    : {len(df_db)} experiments; {len(df_stale)} still out of date")

if len(df_stale):
    display(df_stale[['exp_name', 'date_added', 'source_mtime', 'source_file']])

# ra.purge_experiments('2026-06-04_G')
# ra.purge_experiments(['2026-05-06_E', '2026-05-08_E'])

# Drop only rows that remain stale after populate.
# ra.purge_experiments(df_stale['exp_name'].tolist())

# Wipe the whole database (requires the literal confirmation token).
# ra.purge_database(confirm='YES_DELETE_ALL')

print(f'{len(df_db)} experiments currently in the database.')

/Users/chrischen/opt/anaconda3/envs/retinanalysis/lib/python3.11/site-packages/datajoint/settings.py:979: UserWarning: No datajoint.json found. Using defaults and environment variables. Run `dj.config.save_template()` to create a template configuration.
  config = _create_config()


Ingest source: /Volumes/ChrisNewSSD/single_cell/fred_data/h5
Ingest source: /Volumes/ChrisNewSSD/single_cell/chris_data/h5


Experiments:   0%|          | 0/524 [00:00<?, ?it/s]

Already in database: 2025-12-15_E
Already in database: 2025-12-16_E
Already in database: 2020-02-14_F
Already in database: 2021-01-05_B
Already in database: 2021-01-08_B
Already in database: 2022-01-18_G
Already in database: 2025-01-21_E
Already in database: 2023-01-05_B
Already in database: 2025-01-28_E
Already in database: 2025-01-31_E
Already in database: 2025-02-12_E
Already in database: 2025-02-19_E
Already in database: 2025-03-06_E
Already in database: 2025-04-10_E
Already in database: 2025-04-17_E_2
Already in database: 2025-04-17_E
Already in database: 2025-04-18_E
Already in database: 2025-04-29_E_2
Already in database: 2025-04-29_E
Already in database: 2025-05-06_E
Already in database: 2025-05-13_E
Already in database: 2025-05-14_E_2
Already in database: 2025-05-14_E
Already in database: 2025-05-22_E
Already in database: 2025-05-27_E
Already in database: 2025-05-29_E
Already in database: 2025-05-30_E
Already in database: 2025-06-03_E
Already in database: 2025-06-26_E
Already 

## 3. List single-cell experiments

Experiments are shown in separate `chris_data` and `fred_data` tables. Each protocol gets its own row so a date is easier to scan; repeated experiment, project, and short cell-type values are visually grouped. Owner and species are kept only for the cascading browser and are omitted from the table.

In [4]:
# Section 3 has exactly these visible columns. Owner and species remain
# internal to the cascading browser and are not included here.
df_sc_exps = sc.list_experiments(show=False)

# Accept both the current singular column and an older, comma-joined
# `protocols` column from a module already cached in this kernel.
if 'protocols' in df_sc_exps.columns:
    df_sc_exps['protocol'] = df_sc_exps.pop('protocols').fillna('?').str.split(r',\s*', regex=True)
    df_sc_exps = df_sc_exps.explode('protocol', ignore_index=True)

section3_columns = ['exp_name', 'cell_types', 'protocol']
df_sc_exps = (df_sc_exps.loc[:, section3_columns]
              .drop_duplicates()
              .sort_values(['exp_name', 'protocol'], ignore_index=True))
sc.tree_table(df_sc_exps, levels=['exp_name', 'project', 'cell_types'], height=500)

exp_name,cell_types,protocol
2017-11-21_B,"OFF-transient, ON-alpha",ExpandingSpots
,,EyeMovementTrajectory
2017-12-12_B,"AII, rod bipolar",LedPulse
,,LedPulseFamily
2018-09-06_B,ON-alpha,ChirpStimulus
,,ChirpStimulusLED
,,LedPulse
2018-10-05_B,?,SingleSpot
2019-01-08_B,unknown,LedPulse
,,PulseFamily


'\n<style>\n.ra-tbl { overflow: auto; }\n.ra-tbl table { border-collapse: collapse; font-size: 12.5px;\n                font-variant-numeric: tabular-nums; }\n.ra-tbl th { position: sticky; top: 0; z-index: 1; text-align: left;\n             font-weight: 600; padding: 4px 12px 4px 0;\n             border-bottom: 1px solid rgba(128,128,128,0.6);\n             background: var(--jp-layout-color0, #fff); }\n.ra-tbl td { padding: 2px 12px 2px 0; vertical-align: top;\n             white-space: nowrap; }\n.ra-tbl tr.grp > td { border-top: 1px solid rgba(128,128,128,0.28); }\n.ra-tbl td.num { text-align: right; }\n.ra-tbl td.lead { font-weight: 600; }\n.ra-tbl summary { cursor: pointer; margin: 2px 0; }\n</style>\n<div class="ra-tbl" style="max-height:500px"><table><thead><tr><th>exp_name</th><th>cell_types</th><th>protocol</th></tr></thead><tbody><tr><td class="lead">2017-11-21_B</td><td>OFF-transient, ON-alpha</td><td>ExpandingSpots</td></tr><tr><td class="lead"></td><td></td><td>EyeMovement

### 4.1 Protocol coverage by species

One row per short protocol. Counts are unique experiment dates, not epoch blocks: `primate_dates` and `mouse_dates` show species-specific coverage, and `total_dates` includes every species.


In [6]:
# Scrollable protocol inventory, sorted by the number of dates.
df_protocol_inventory = sc.protocol_inventory(height=500)


134 protocols across 527 experiment dates.


protocol,primate_dates,mouse_dates,total_dates
SingleSpot,243,79,329
ExpandingSpots,258,57,325
SplitFieldCentering,235,37,280
VariableMeanNoise,121,16,140
LedPulse,85,38,125
ContrastReversingGrating,45,51,97
DovesMovie,57,1,59
LinearEquivalentDisc,21,37,58
JitteredNoise,46,0,51
LedNoiseFamily,40,7,48


## 4. Find experiments by protocol

Search protocol names case-insensitively. The returned DataFrame remains one row per epoch block. The expanded block table shows fixed NDF settings plus the actual numeric filter-wheel reading for every block; an embedded `FWx` label is not used in place of the wheel reading.

In [9]:
df_blocks = sc.find_blocks('LinearEquivalentAnnulus')

379 blocks | 37 experiments | 3 protocol(s) matching 'LinearEquivalentAnnulus'


exp_name,blocks,protocols,block_ids
2022-07-05_G,28,edu.washington.riekelab.turner.unused.LinearEquivalentAnnulus,"23036-23039, 23054-23055, 23061-23064, 23079-23082, 23085-23088, 23152-23153, 23161-23162, 23172-23175, 23203-23204"
2022-07-12_G,12,edu.washington.riekelab.turner.unused.LinearEquivalentAnnulus,"23302-23304, 23319-23320, 23323, 23329-23330, 23332-23333, 23340-23341"
2022-08-09_G,9,edu.washington.riekelab.turner.unused.LinearEquivalentAnnulus,"23352-23354, 23362-23364, 23396-23397, 23412"
2022-09-09_G,3,edu.washington.riekelab.turner.unused.LinearEquivalentAnnulus,"23749-23750, 23757"
2022-11-23_G,7,edu.washington.riekelab.turner.unused.LinearEquivalentAnnulus,"24254-24257, 24269-24271"
2022-12-01_G,8,edu.washington.riekelab.turner.unused.LinearEquivalentAnnulus,"24405-24410, 24414-24415"
2023-04-18_B,13,edu.washington.riekelab.turner.unused.LinearEquivalentAnnulus,"20404-20406, 20420-20421, 20431-20433, 20464-20466, 20475-20476"
2023-04-18_B_2,8,edu.washington.riekelab.turner.unused.LinearEquivalentAnnulus,"20357, 20371, 20373-20374, 20382-20383, 20389, 20396"
2023-05-02_B,28,edu.washington.riekelab.turner.unused.LinearEquivalentAnnulus,"20569-20570, 20575-20576, 20581-20582, 20658-20661, 20666, 20668-20669, 20674-20675, 20678-20680, 20685-20686, 20691-20694, 20701, 20703-20705"
2023-05-11_B,27,edu.washington.riekelab.turner.unused.LinearEquivalentAnnulus,"20792-20794, 20798-20799, 20804, 20806, 20815-20816, 20827-20830, 20835, 20837, 20845-20846, 20856-20861, 20868, 20872, 20885-20886"


exp_name,protocol,block_id,NDF + FW
2022-07-05_G,edu.washington.riekelab.turner.unused.LinearEquivalentAnnulus,23036,EL1 + EL2 + FW0
2022-07-05_G,edu.washington.riekelab.turner.unused.LinearEquivalentAnnulus,23037,EL1 + EL2 + FW0
2022-07-05_G,edu.washington.riekelab.turner.unused.LinearEquivalentAnnulus,23038,EL1 + EL2 + FW0
2022-07-05_G,edu.washington.riekelab.turner.unused.LinearEquivalentAnnulus,23039,EL1 + EL2 + FW0
2022-07-05_G,edu.washington.riekelab.turner.unused.LinearEquivalentAnnulus,23054,EL1 + EL2 + FW0
2022-07-05_G,edu.washington.riekelab.turner.unused.LinearEquivalentAnnulus,23055,EL1 + EL2 + FW0
2022-07-05_G,edu.washington.riekelab.turner.unused.LinearEquivalentAnnulus,23061,EL1 + EL2 + FW0
2022-07-05_G,edu.washington.riekelab.turner.unused.LinearEquivalentAnnulus,23062,EL1 + EL2 + FW0
2022-07-05_G,edu.washington.riekelab.turner.unused.LinearEquivalentAnnulus,23063,EL1 + EL2 + FW0
2022-07-05_G,edu.washington.riekelab.turner.unused.LinearEquivalentAnnulus,23064,EL1 + EL2 + FW0


## 5. Browse and summarize experiments

Use the cascading menus to select data owner, species, and experiment. Each experiment date appears once in the menu, regardless of how many protocols ran that day. The overview is organized as cell → epoch group (group label) → protocol, with block and epoch counts. Then select an epoch block and click **Load original traces** to read and display every unprocessed Amp1 epoch trace from the H5 file.

In [ ]:
# Pass one row per date to the browser; Section 3 intentionally has one
# row per protocol and must not duplicate dates in this menu.
browser_dates = df_sc_exps[['exp_name']].drop_duplicates(ignore_index=True)
experiment_browser = sc.summarize_experiments(browser_dates)